# Predicting Steam Game Popularity

This notebook investigates whether public Steam metadata can predict peak concurrent players (CCU). CCU is an imperfect proxy for popularity: it does not measure sales, revenue, marketing impact, or long-term player retention.

The core evaluation uses a small, interpretable feature set: genre, platform availability, and tag count. An optional GAN experiment creates synthetic records only in memory and is evaluated separately from the main result.

## Setup

Install the pinned dependencies with `python -m pip install -r requirements.txt`. Download `games_may2024_full.csv` separately and place it in `BestBet/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

RANDOM_STATE = 42
DATA_PATH = Path('BestBet/games_may2024_full.csv')

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset not found at {DATA_PATH}. See the README for download instructions.'
    )

sns.set_theme(style='whitegrid')

## Load and prepare the data

The target is transformed with `log1p` because CCU has a long right tail. This makes RMSE more stable, but results are therefore reported in log-CCU units.

In [ ]:
steam_games = pd.read_csv(DATA_PATH)

required_columns = {'genres', 'tags', 'windows', 'mac', 'linux', 'peak_ccu'}
missing_columns = required_columns - set(steam_games.columns)
if missing_columns:
    raise ValueError(f'Dataset is missing required columns: {sorted(missing_columns)}')

df = steam_games.loc[:, list(required_columns)].copy()
df = df.dropna(subset=['peak_ccu'])
df['genres'] = df['genres'].fillna('Unknown').astype(str)
df['tag_count'] = df['tags'].fillna('').astype(str).str.count(',').add(1)
df.loc[df['tags'].fillna('').eq(''), 'tag_count'] = 0

feature_columns = ['genres', 'windows', 'mac', 'linux', 'tag_count']
X = df[feature_columns].copy()
X[['windows', 'mac', 'linux']] = X[['windows', 'mac', 'linux']].fillna(False).astype(int)
y = np.log1p(df['peak_ccu'].clip(lower=0))

print(f'Rows analysed: {len(df):,}')
print(f'Median peak CCU: {df.peak_ccu.median():,.0f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(y, bins=60, ax=axes[0])
axes[0].set(title='Distribution of log1p(peak CCU)', xlabel='log1p(peak CCU)')
sns.boxplot(data=df, x='mac', y='peak_ccu', showfliers=False, ax=axes[1])
axes[1].set(title='Peak CCU by macOS support', xlabel='macOS supported', ylabel='Peak CCU')
plt.tight_layout()

## Evaluate a baseline and XGBoost

The dummy baseline predicts the training-set mean. Beating it demonstrates that the features add signal. XGBoost is suitable for this tabular data; no sequence models are used because the feature columns have no meaningful order.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[('genre', OneHotEncoder(handle_unknown='ignore'), ['genres'])],
    remainder='passthrough',
)

models = {
    'Mean baseline': Pipeline([('preprocess', preprocessor), ('model', DummyRegressor())]),
    'XGBoost': Pipeline([
        ('preprocess', preprocessor),
        ('model', XGBRegressor(
            n_estimators=200, max_depth=8, learning_rate=0.047, subsample=0.96,
            objective='reg:squarederror', random_state=RANDOM_STATE, n_jobs=-1
        )),
    ]),
}

cv = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
scores = {}
for name, model in models.items():
    rmse = -cross_val_score(model, X, y, cv=cv, scoring='neg_root_mean_squared_error')
    scores[name] = {'mean_rmse': rmse.mean(), 'std_rmse': rmse.std()}

results = pd.DataFrame(scores).T.sort_values('mean_rmse')
results

## Optional GAN augmentation experiment

Set `RUN_GAN_EXPERIMENT = True` only to investigate augmentation. The GAN is fit **only after the holdout split and only on training rows**, so generated data cannot leak information from the evaluation set. It creates records in memory and does not save a CSV. 

In [ ]:
RUN_GAN_EXPERIMENT = False
GAN_EPOCHS = 300
SYNTHETIC_FRACTION = 0.10

def augment_training_data_with_gan(X_train, y_train, fraction=SYNTHETIC_FRACTION, epochs=GAN_EPOCHS):
    """Create an in-memory synthetic training set; never use validation or test rows."""
    import tensorflow as tf
    from sklearn.preprocessing import MinMaxScaler

    if not 0 < fraction <= 1:
        raise ValueError('fraction must be in the interval (0, 1].')

    genre_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    genre_matrix = genre_encoder.fit_transform(X_train[['genres']])
    numeric_columns = ['windows', 'mac', 'linux', 'tag_count']
    numeric_matrix = X_train[numeric_columns].to_numpy(dtype=float)
    target_matrix = y_train.to_numpy(dtype=float).reshape(-1, 1)
    raw_matrix = np.hstack([genre_matrix, numeric_matrix, target_matrix])

    scaler = MinMaxScaler()
    training_matrix = scaler.fit_transform(raw_matrix).astype('float32')
    feature_dim, latent_dim = training_matrix.shape[1], 32

    generator = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(latent_dim,)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(feature_dim, activation='sigmoid'),
    ])
    discriminator = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(feature_dim,)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])
    discriminator.compile(optimizer='adam', loss='binary_crossentropy')

    discriminator.trainable = False
    gan_input = tf.keras.Input(shape=(latent_dim,))
    gan = tf.keras.Model(gan_input, discriminator(generator(gan_input)))
    gan.compile(optimizer='adam', loss='binary_crossentropy')
    discriminator.trainable = True

    batch_size = min(256, len(training_matrix))
    for _ in range(epochs):
        real_index = np.random.randint(0, len(training_matrix), batch_size)
        real_batch = training_matrix[real_index]
        noise = np.random.normal(size=(batch_size, latent_dim)).astype('float32')
        fake_batch = generator.predict(noise, verbose=0)
        discriminator.train_on_batch(real_batch, np.ones((batch_size, 1)))
        discriminator.train_on_batch(fake_batch, np.zeros((batch_size, 1)))
        noise = np.random.normal(size=(batch_size, latent_dim)).astype('float32')
        gan.train_on_batch(noise, np.ones((batch_size, 1)))

    sample_count = max(1, int(len(X_train) * fraction))
    synthetic_scaled = generator.predict(
        np.random.normal(size=(sample_count, latent_dim)), verbose=0
    )
    synthetic = scaler.inverse_transform(synthetic_scaled)

    genre_count = genre_matrix.shape[1]
    genre_labels = genre_encoder.categories_[0][synthetic[:, :genre_count].argmax(axis=1)]
    numeric = synthetic[:, genre_count:-1]
    synthetic_X = pd.DataFrame({
        'genres': genre_labels,
        'windows': (numeric[:, 0] >= 0.5).astype(int),
        'mac': (numeric[:, 1] >= 0.5).astype(int),
        'linux': (numeric[:, 2] >= 0.5).astype(int),
        'tag_count': np.clip(np.rint(numeric[:, 3]), 0, None).astype(int),
    })
    synthetic_y = pd.Series(np.clip(synthetic[:, -1], 0, None), name=y_train.name)
    return pd.concat([X_train, synthetic_X], ignore_index=True), pd.concat([y_train.reset_index(drop=True), synthetic_y], ignore_index=True)


## Inspect holdout predictions

This final check fits the best tabular model on a training split and compares predictions against unseen games. Treat any result as an estimate conditioned on this dataset, rather than a deployable measure of a game's commercial prospects.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
if RUN_GAN_EXPERIMENT:
    X_train_fit, y_train_fit = augment_training_data_with_gan(X_train, y_train)
    print(f'GAN augmentation enabled: {len(X_train_fit) - len(X_train):,} synthetic training rows added.')
else:
    X_train_fit, y_train_fit = X_train, y_train

final_model = models['XGBoost'].fit(X_train_fit, y_train_fit)
predictions = final_model.predict(X_test)
holdout_rmse = mean_squared_error(y_test, predictions, squared=False)
print(f'Holdout RMSE (log1p peak CCU): {holdout_rmse:.3f}')

plt.figure(figsize=(6, 6))
plt.scatter(y_test, predictions, alpha=0.15, s=12)
limits = [min(y_test.min(), predictions.min()), max(y_test.max(), predictions.max())]
plt.plot(limits, limits, 'r--', label='Perfect prediction')
plt.xlabel('Observed log1p(peak CCU)')
plt.ylabel('Predicted log1p(peak CCU)')
plt.title('XGBoost predictions on the holdout set')
plt.legend()
plt.show()

## Interpretation and limitations

A model that performs better than the mean baseline has found associations in the available metadata. It has not established that genre, platform support, or tag count *cause* popularity. These features can act as proxies for unobserved factors such as budget, marketing, and studio capacity.

Possible next steps are a time-aware evaluation split, stronger publication-date features, and a documented experiment that tests whether augmentation improves holdout performance.